# 02 — Preprocessing (part A: cleaning)
MAS Lab 2 — Fake news detection (WCO + MOA)

Implements the decisions taken from the exploration (notebook 01):
duplicates, short texts, text noise, useless columns, image validity, shortcut check, stratified split.
Output: a clean table + a fixed train/test split reused by all later notebooks.

## 0. Setup and data (same download as notebook 01, ~2 min)

In [1]:
!pip install -q gdown
import os, re, html, glob, zipfile, json
import numpy as np
import pandas as pd
import gdown

WORK = '/kaggle/working/finefake'
OUT  = '/kaggle/working/clean'
os.makedirs(WORK, exist_ok=True); os.makedirs(OUT, exist_ok=True)
SEED = 42

archive = os.path.join(WORK, 'finefake_archive')
pkl = os.path.join(WORK, 'FineFake.pkl')
if not os.path.exists(pkl):
    if not os.path.exists(archive):
        gdown.download(id='16D9ix7ZOisa4VVBznBTBcv1N7TA-jodH', output=archive, quiet=False)
    with zipfile.ZipFile(archive) as z:
        z.extractall(WORK)
    os.remove(archive)

df = pd.read_pickle(pkl)
print('Raw shape:', df.shape)

Raw shape: (16909, 13)


## 1. Drop useless columns (finding #8)
`author`, `date`, `comment` are mostly empty; `description`, `relation`, `entity_id` are not used.

In [2]:
df = df.drop(columns=['author', 'date', 'comment', 'description', 'relation', 'entity_id'])
df = df.rename(columns={'fine-grained label': 'fine_label'})
print(df.columns.tolist())

['text', 'image_path', 'topic', 'fine_label', 'label', 'knowledge_embedding', 'platform']


## 2a. Text cleaning (findings #4, #5, #6, #7)
Only noise is removed. No lowercasing, stopword removal or stemming: sentence encoders work better on natural text.

In [3]:
URL_RE     = re.compile(r'https?://\S+|www\.\S+')
MENTION_RE = re.compile(r'@\w+')
HASHTAG_RE = re.compile(r'#(\w+)')
HTML_RE    = re.compile(r'<[^>]+>')
PREFIX_RE  = re.compile(r'^\s*(TIL( that)?|PsBattle:|\[OC\]|\[HAFF\]|ELI5:?|FunFact:?)\s*', re.IGNORECASE)
QUOTES = {'\u2019': "'", '\u2018': "'", '\u201c': '"', '\u201d': '"', '\u2013': '-', '\u2014': '-'}

BOILER_RE = [
    re.compile(r'(?:(?<!\d)1\.)?\s*How relevant is this ad to you\?.*?Other issues', re.DOTALL),
    re.compile(r'Ad never loaded.*?Other issues', re.DOTALL),
    re.compile(r'Cancel Submit Thank You!.*?Close', re.DOTALL),
    re.compile(r'Breaking News, Latest News and Videos'),
]

def clean_text(t):
    t = str(t)
    t = html.unescape(t)
    t = HTML_RE.sub(' ', t)
    for k, v in QUOTES.items():
        t = t.replace(k, v)
    t = URL_RE.sub(' [URL] ', t)
    t = MENTION_RE.sub(' ', t)
    t = HASHTAG_RE.sub(r'\1', t)
    t = re.sub(r'\s+', ' ', t).strip()
    for b in BOILER_RE:
        t = b.sub(' ', t)
    t = PREFIX_RE.sub('', t)
    t = re.sub(r'\s+', ' ', t).strip()
    return t

df['text_clean'] = df['text'].apply(clean_text)

for i in df[df['text'].str.contains('http|#|TIL|\n', regex=True)].sample(4, random_state=SEED).index:
    print('BEFORE:', repr(df.at[i, 'text'][:150]))
    print('AFTER :', repr(df.at[i, 'text_clean'][:150])); print()

BEFORE: 'How the Split Screen Framed Trump and ClintonThe split screen told the story at the first 2016 presidential debate, and it was not kind to Donald J. T'
AFTER : 'How the Split Screen Framed Trump and ClintonThe split screen told the story at the first 2016 presidential debate, and it was not kind to Donald J. T'

BEFORE: '\nA meme accurately depicts a stack of books as “the most banned books from public libraries and schools in the US,” including books like “To Kill a Mo'
AFTER : 'A meme accurately depicts a stack of books as "the most banned books from public libraries and schools in the US," including books like "To Kill a Moc'

BEFORE: '\nA map shows the portions of the U.S. that have the greatest incidence of human trafficking.'
AFTER : 'A map shows the portions of the U.S. that have the greatest incidence of human trafficking.'

BEFORE: 'Breaking News, Latest News and Videos1. How relevant is this ad to you?\n\nVideo player was slow to load content Video content never load

### 2b. Check for other source-revealing prefixes
Most common first words of Reddit texts, **before** and **after** cleaning.
If a tag (e.g. `PSA:`, `[Image]`) is still frequent after cleaning, add it to `PREFIX_RE` and rerun the cell above.

In [4]:
reddit = df[df['platform'] == 'reddit']

first_raw   = reddit['text'].astype(str).str.strip().str.split().str[0]
first_clean = reddit['text_clean'].str.split().str[0]

print(pd.DataFrame({
    'before': first_raw.value_counts().head(15).index,
    'n_before': first_raw.value_counts().head(15).values,
    'after': first_clean.value_counts().head(15).index,
    'n_after': first_clean.value_counts().head(15).values,
}))
print('\nReddit texts:', len(reddit))

       before  n_before after  n_after
0   PsBattle:      1670  This      268
1      [HAFF]       392     A      251
2         The       128   The      247
3           A       101   Cat      116
4        This        48  this       71
5          My        45     a       50
6           I        28    In       48
7        What        23   Man       48
8         How        21    My       47
9         TIL        21   Dog       42
10        New        19   cat       41
11         In        18     I       28
12        Why        17  Baby       27
13         An        16   New       24
14  Democrats        15  What       23

Reddit texts: 4048


## 3. Remove empty and very short texts (finding #2)

In [5]:
df['n_words'] = df['text_clean'].str.split().str.len()
before = len(df)
df = df[df['n_words'] >= 3].copy()
print(f'Removed {before - len(df)} texts with < 3 words -> {len(df)} rows left')

Removed 284 texts with < 3 words -> 16625 rows left


## 4. Remove duplicates (finding #1)
Done on the *cleaned* text, so texts that differ only by a link or spaces are also caught.
If the same text has **both** labels, it is a labelling conflict -> all copies are dropped.

In [6]:
n_labels = df.groupby('text_clean')['label'].nunique()
conflicts = n_labels[n_labels > 1].index
print('Texts with conflicting labels:', len(conflicts))

before = len(df)
df = df[~df['text_clean'].isin(conflicts)]
df = df.drop_duplicates(subset='text_clean', keep='first')
df = df.reset_index(drop=True)
print(f'Removed {before - len(df)} rows -> {len(df)} rows left')

Texts with conflicting labels: 21
Removed 1327 rows -> 15298 rows left


## 5. Image validity (findings #9, #10)
Check the **full path** (fixes the file-name-only check of notebook 01), the extension, and that the image really opens.
`.ico` favicons, missing, broken or tiny images -> treated as *no image*.

In [7]:
from PIL import Image
from tqdm.auto import tqdm

df['image_full'] = df['image_path'].apply(lambda p: os.path.join(WORK, str(p)))
df['img_ext'] = df['image_path'].astype(str).str.extract(r'(\.[A-Za-z0-9]+)$')[0].str.lower()
print('Image extensions:'); print(df['img_ext'].value_counts(dropna=False))

def image_ok(path, ext):
    if ext == '.ico' or not os.path.exists(path):
        return False
    try:
        with Image.open(path) as im:
            w, h = im.size
            return min(w, h) >= 32
    except Exception:
        return False

df['has_image'] = [image_ok(p, e) for p, e in tqdm(zip(df['image_full'], df['img_ext']), total=len(df))]
print('\nRows with a valid image:', df['has_image'].sum(), '/', len(df))

df.loc[~df['has_image'], 'image_full'] = None
df = df.drop(columns=['img_ext', 'image_path'])
print('Null image paths:', df['image_full'].isna().sum())

Image extensions:
img_ext
.jpeg    13161
.png      1985
.ico       135
.gif        17
Name: count, dtype: int64



Rows with a valid image: 15163 / 15298
Null image paths: 135


## 6. Shortcut check (finding #11): is the label predictable from the platform alone?

In [8]:
tab = df.groupby('platform')['label'].agg(n='size', real_ratio='mean').round(3)
print(tab.sort_values('real_ratio'))
print('\nValid image ratio by label:'); print(df.groupby('label')['has_image'].mean().round(3))

                   n  real_ratio
platform                        
snope           7556       0.327
reddit          2700       0.349
twitter          854       0.377
washingtonpost   996       0.679
cnn             2250       0.896
apnews           707       0.908
cdc_gov          235       0.987

Valid image ratio by label:
label
0    1.000
1    0.982
Name: has_image, dtype: float64


## 7. Final distribution + stratified train/test split (finding #12)
80% train (used inside the metaheuristic with cross-validation) / 20% test (touched only once, at the end).

In [9]:
from sklearn.model_selection import train_test_split

print('Final shape:', df.shape)
print('\nLabels:'); print(df['label'].value_counts())
print('\nFine labels:'); print(df['fine_label'].value_counts())
print('\nWords (clean):'); print(df['n_words'].describe().round(1))

idx_train, idx_test = train_test_split(df.index, test_size=0.2, stratify=df['label'], random_state=SEED)
df['split'] = 'train'
df.loc[idx_test, 'split'] = 'test'
print('\nSplit x label:'); print(pd.crosstab(df['split'], df['label']))

Final shape: (15298, 10)

Labels:
label
0    7993
1    7305
Name: count, dtype: int64

Fine labels:
fine_label
0    7305
3    3418
4    1633
1    1399
2    1266
5     277
Name: count, dtype: int64

Words (clean):
count    15298.0
mean       229.7
std        662.0
min          3.0
25%         13.0
50%         21.0
75%        145.0
max      18320.0
Name: n_words, dtype: float64

Split x label:
label     0     1
split            
test   1599  1461
train  6394  5844


## 8. Save

In [10]:
keep = ['text_clean', 'image_full', 'has_image', 'label', 'fine_label',
        'topic', 'platform', 'knowledge_embedding', 'split']
df[keep].to_pickle(os.path.join(OUT, 'finefake_clean.pkl'))
print('Saved:', os.path.join(OUT, 'finefake_clean.pkl'), df[keep].shape)

Saved: /kaggle/working/clean/finefake_clean.pkl (15298, 9)


## 9. Final checks (boilerplate and prefixes removed)

In [13]:
print('Texts still starting with [HAFF]:', df['text_clean'].str.startswith('[HAFF]').sum())

for phrase in ['How relevant is this ad to you', 'Video player was slow',
               'Ad never loaded', 'Thank You! Your effort', 'Breaking News, Latest News and Videos']:
    n = df['text_clean'].str.contains(phrase, regex=False).sum()
    print(f'{phrase!r}: {n} texts')

mask = df['text_clean'].str.contains('How relevant is this ad to you', regex=False)
if mask.any():
    t = df[mask]['text_clean'].iloc[0]
    k = t.find('How relevant')
    print('\n', repr(t[max(0, k - 150): k + 900]))

Texts still starting with [HAFF]: 0
'How relevant is this ad to you': 0 texts
'Video player was slow': 0 texts
'Ad never loaded': 0 texts
'Thank You! Your effort': 0 texts
'Breaking News, Latest News and Videos': 0 texts
